In [25]:
import pandas as pd
import numpy as np
import sqlite3
import statsmodels.stats.multitest as mt
from scipy import stats

In [5]:
with sqlite3.connect("../data/pancreas_proteomics.db") as conn:
    df = pd.read_sql("SELECT * FROM imputed_MNAR", conn)

#df
df_pivot = df.pivot(index='Protein ID', columns='Sample ID', values='Intensity')
df_pivot

Sample ID,sample_01,sample_02,sample_03,sample_05,sample_06,sample_07,sample_08,sample_09,sample_10,sample_11,sample_12,sample_13,sample_15,sample_16,sample_17,sample_18,sample_19,sample_20
Protein ID,,,,,,,,,,,,,,,,,,
A0A0B4J1X5,18.394000,18.399589,12.708169,18.465326,18.636132,18.717648,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169,12.708169
A0A0B4J2D5,20.092054,19.930175,19.702742,19.849352,20.007990,19.487028,19.507953,20.218176,19.303957,19.674433,19.492888,19.129642,19.767037,19.075852,20.177990,19.835756,19.091262,19.535666
A0A2R8Y4L2,20.977451,20.504533,20.650515,21.236394,20.860099,21.425081,21.995113,20.791442,20.291698,20.292409,20.601955,20.187754,20.624972,20.821667,20.661272,20.753543,20.658981,20.795377
A0A2R8Y619,21.592487,22.053037,21.991385,21.699182,21.865964,12.708169,21.883938,21.363115,21.471785,12.708169,21.501527,12.708169,12.708169,21.306108,12.708169,20.767906,12.708169,21.431007
A0AV96,18.866362,19.257156,19.330834,18.495888,19.004611,19.069323,19.312721,19.124717,19.295020,18.891364,18.636439,18.656816,18.911594,18.977309,19.027774,19.167592,18.811695,18.973236
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Q9Y6W3,17.004410,17.079889,16.721334,16.851973,16.751962,16.818081,16.926340,16.782336,16.786909,16.681789,16.956898,16.682235,16.781762,16.565844,16.975755,16.914065,16.772178,12.708169
Q9Y6W5,17.392511,17.423847,16.625088,17.753186,17.362170,12.708169,17.355910,17.058247,12.708169,16.847490,17.271432,16.744053,12.708169,17.578659,12.708169,17.730598,17.765363,17.554511
Q9Y6X5,17.420279,16.293992,17.874419,17.893090,16.879647,12.708169,16.629186,16.605248,16.486064,17.024385,16.550013,16.843545,16.841090,17.550291,16.973724,17.949008,12.708169,17.522159


In [ ]:
results = []
for protein in df_pivot.index:
    index = df_pivot.index.get_loc(protein)
    t1d = df_pivot.iloc[index, list(range(9))].dropna()
    ctrl = df_pivot.iloc[index, list(range(9,18))].dropna() 
    #print(ctrl)

    t_stat, p_val = stats.ttest_ind(t1d, ctrl, equal_var=False)
    fc = t1d.mean() - ctrl.mean()

    results.append((protein, p_val, fc))

results_df = pd.DataFrame(results, columns=['Protein ID', 'Log2FC', 'p_value']).dropna()
_, results_df['adj_p_value'], _, _ = mt.multipletests(results_df['p_value'], method='fdr_bh')

    

c:\Users\amika\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\scipy\stats\_axis_nan_policy.py:601: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


(array([False, False, False, ...,  True, False, False], shape=(4959,)),
 array([ 1.        ,  0.50682862,  0.62216447, ..., -0.62529722,
         0.97567791,  0.09507026], shape=(4959,)),
 np.float64(1.034342188199755e-05),
 1.0082677959265981e-05)

In [21]:
list(range(9,18))

[9, 10, 11, 12, 13, 14, 15, 16, 17]